## Loading Data

In [1]:
# loading datasets
import pandas as pd
data_w_target = pd.read_csv("C:/Users/aiden/OneDrive/Desktop/School/6380/GroupProject/list_of_compounds_for_training.csv")
data_for_testing = pd.read_csv("C:/Users/aiden/OneDrive/Desktop/School/6380/GroupProject/list_of_compounds_for_computational_screening.csv")
data_w_target


,Name,senolytic,Library,Source,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed
0,Azaguanine-8,0,Prestwick,Not identified,c12/N=C(\NC(c1nn[nH]2)=O)/N,3.024307,441.024163,7.844935,5.327239,5.327239,...,0,0,0,0,0,0,0,0,0,0.430316
1,Allantoin,0,Prestwick,Not identified,N1C(NC(C1=O)NC(=O)N)=O,2.534439,225.377060,8.430721,5.379445,5.379445,...,0,0,0,0,0,0,0,0,2,0.325138
2,Acetazolamide,0,Prestwick,Not identified,c1(S(=O)(=O)N)sc(nn1)NC(=O)C,2.938691,422.352468,10.060478,6.513019,8.146012,...,1,0,0,0,0,0,0,0,0,0.631859
3,Metformin hydrochloride,0,Prestwick,Not identified,C(NC(=N)N)(=N)N(C)C,3.644486,126.919685,7.439158,5.524564,5.524564,...,0,0,0,0,0,0,0,0,0,0.248785
4,Atracurium besylate,0,Prestwick,Not identified,[N+]1(C(c2c(cc(c(c2)OC)OC)CC1)Cc1cc(c(cc1)OC)O...,0.987040,2158.836594,48.141042,41.328212,41.328212,...,0,0,0,0,0,0,0,4,0,0.038349
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2518,Curcumin,1,"GPNCL, ENZO","Source 12 - Yousefzadeh et al, 2018",COC1=C(C=CC(=C1)/C=C/C(=O)CC(=O)/C=C/C2=CC(=C(...,1.958861,822.040000,19.811190,15.008030,15.008030,...,0,0,0,0,0,0,0,0,0,0.548123
2519,Dasatinib,1,"Unknown library, see publication source","Source 13 - Zhu et al, 2015",CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,1.431593,1111.432171,23.371668,18.507135,20.079560,...,0,0,0,0,1,0,0,0,0,0.465717
2520,Navitoclax,1,"Unknown library, see publication source","Source 14 - Zhu et al, 2016",CC1(CCC(=C(C1)CN2CCN(CC2)C3=CC=C(C=C3)C(=O)NS(...,1.017180,2532.551918,46.408991,36.449290,39.654708,...,1,1,0,0,0,0,0,0,0,0.104649
2521,A1331852,1,"Unknown library, see publication source","Source 15 - Zhu et al, 2017",O=C(NC1=NC(C=CC=C2)=C2S1)C3=C(CN(C4=CC=C(C5=C(...,0.969918,2030.733706,32.569974,26.984648,27.801144,...,0,0,0,0,1,0,0,0,0,0.185260


## Combining dataframes

In [2]:
# concating dataframes
data_without_target = data_w_target.drop("senolytic",axis = 1)
combined_datasets = pd.concat([data_without_target, data_for_testing])


## Removing Duplicates

In [3]:
# removing duplicate values
combined_datasets = combined_datasets.drop_duplicates(subset="SMILES")

## Using Clustering to select senolytic candidates
On only the labeled data, I use SMOTE to balance the data. Then using a L1 penalty on the logistic regression model, select my features (Lasso Regression). I use the selected features to fit a logistic regression model with the SMOTE data. This logistic regression model gives output in terms of log-odds. This is better for my clustering later on because the range is larger (better geometrically). Next I return to the full dataset before smote is applied. I selected 13 principal components. These are the 13 axes that capture the largest amount of variance from the data. Each compound is represented by its coordinates along the 13 principal components. Using my logistic model, I predict on the whole dataset. I use the log odds + the 13 principal component coordinates in order to cluster the entire dataset. Then, looking at only the labeled data in each cluster, decide which cluster is the most dense in senolytics. From there, I look at the unlabeled data in the cluster, and look at the ones with the greatest log-odds. These would be my most likely candidates for possible senolytics.

In [6]:
# SMOTE BEFORE LASSO (used once and reused for final logistic)
# clustering selects 13 principal components from full dataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import fisher_exact
from imblearn.over_sampling import SMOTE

# --------------------------------------------------
# STEP 1: Prepare FULL dataset (keep unlabeled)
# --------------------------------------------------

label_df = data_w_target[["Name", "senolytic"]].copy()

full_df = combined_datasets.copy()
full_df = full_df.merge(label_df, on="Name", how="left")
full_df["senolytic"] = pd.to_numeric(full_df["senolytic"], errors="coerce")

# Separate numeric features
feature_cols = full_df.select_dtypes(include=[np.number]).columns.tolist()
if "senolytic" in feature_cols:
    feature_cols.remove("senolytic")

X_all = full_df[feature_cols].values

# --------------------------------------------------
# STEP 2: Remove low variance + highly correlated features
# --------------------------------------------------

variances = np.var(X_all, axis=0)
keep_var = variances > 1e-6
X_all = X_all[:, keep_var]
feature_cols = np.array(feature_cols)[keep_var]

corr_matrix = np.corrcoef(X_all, rowvar=False)
to_drop = set()

for i in range(corr_matrix.shape[0]):
    for j in range(i + 1, corr_matrix.shape[1]):
        if abs(corr_matrix[i, j]) > 0.95:
            to_drop.add(j)

keep_idx = [i for i in range(X_all.shape[1]) if i not in to_drop]
X_all = X_all[:, keep_idx]
feature_cols = feature_cols[keep_idx]

# --------------------------------------------------
# STEP 3: Standardize (fit on ALL data)
# --------------------------------------------------

scaler = StandardScaler()
X_scaled_all = scaler.fit_transform(X_all)

# --------------------------------------------------
# STEP 4: Apply SMOTE ONCE (before LASSO)
# --------------------------------------------------

labeled_mask = ~full_df["senolytic"].isna()
X_labeled = X_scaled_all[labeled_mask]
y_labeled = full_df.loc[labeled_mask, "senolytic"].values.astype(int)

print("Original class distribution:", np.bincount(y_labeled))

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_labeled, y_labeled)

print("Class distribution AFTER SMOTE:", np.bincount(y_balanced))

# --------------------------------------------------
# STEP 5: LASSO Feature Selection (on SMOTE data)
# --------------------------------------------------

lasso_logreg = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=5000,
    random_state=42
)

lasso_logreg.fit(X_balanced, y_balanced)

coef_mask = lasso_logreg.coef_.flatten() != 0
selected_features = feature_cols[coef_mask]

print("\nSelected Features From LASSO:\n")
for f in selected_features:
    print(f)

# Reduce both datasets to selected features
X_scaled_selected = X_scaled_all[:, coef_mask]
X_balanced_selected = X_balanced[:, coef_mask]

# --------------------------------------------------
# STEP 6: Final Logistic Model (NO additional SMOTE)
# --------------------------------------------------

final_logreg = LogisticRegression(
    max_iter=5000,
    random_state=42
)

final_logreg.fit(X_balanced_selected, y_balanced)

# Use decision function (better geometry for clustering)
senolytic_axis_all = final_logreg.decision_function(X_scaled_selected)

full_df["SenolyticAxis"] = senolytic_axis_all

# --------------------------------------------------
# STEP 7: PCA on FULL cleaned feature space
# --------------------------------------------------

pca = PCA(n_components=13, random_state=42)
X_pca_all = pca.fit_transform(X_scaled_all)

# Combine supervised axis + structural PCs
X_embed_all = np.column_stack([
    senolytic_axis_all,
    X_pca_all
])

# --------------------------------------------------
# STEP 8: Cluster ALL compounds
# --------------------------------------------------

k = 13
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
clusters_all = kmeans.fit_predict(X_embed_all)

full_df["Cluster"] = clusters_all

# --------------------------------------------------
# STEP 9: Compute enrichment (labeled only)
# --------------------------------------------------

total_positive = np.sum(y_labeled)
total_negative = len(y_labeled) - total_positive

print("\nCluster Enrichment Summary\n")

cluster_enrichment = {}

for c in range(k):
    cluster_subset = full_df[full_df["Cluster"] == c]
    labeled_cluster = cluster_subset[~cluster_subset["senolytic"].isna()]

    cluster_size = len(labeled_cluster)
    pos_in_cluster = np.sum(labeled_cluster["senolytic"] == 1)
    neg_in_cluster = cluster_size - pos_in_cluster

    if cluster_size == 0:
        continue

    table = [
        [pos_in_cluster, neg_in_cluster],
        [total_positive - pos_in_cluster, total_negative - neg_in_cluster]
    ]

    odds_ratio, p_value = fisher_exact(table)

    enrichment = (
        (pos_in_cluster / cluster_size) /
        (total_positive / len(y_labeled))
    )

    cluster_enrichment[c] = enrichment

    print(f"Cluster {c}")
    print(f" Labeled size: {cluster_size}")
    print(f" Positives: {pos_in_cluster}")
    print(f" Enrichment Ratio: {enrichment:.3f}")
    print(f" Odds Ratio: {odds_ratio:.3f}")
    print(f" Fisher p-value: {p_value:.6f}")
    print("-" * 40)

# --------------------------------------------------
# STEP 10: Find MOST enriched cluster
# --------------------------------------------------

best_cluster = max(cluster_enrichment, key=cluster_enrichment.get)
print(f"\nMost Enriched Cluster: {best_cluster}")

# --------------------------------------------------
# STEP 11: Rank UNKNOWN compounds in best cluster
# --------------------------------------------------

unknown_in_best = full_df[
    (full_df["Cluster"] == best_cluster) &
    (full_df["senolytic"].isna())
]

unknown_ranked = unknown_in_best.sort_values(
    by="SenolyticAxis",
    ascending=False
)

print("\nTop Unknown Compounds To Test:\n")
print(unknown_ranked[["Name", "SenolyticAxis"]].head(20))

Original class distribution: [3233   58]
Class distribution AFTER SMOTE: [3233 3233]

Selected Features From LASSO:

BalabanJ
BertzCT
EState_VSA1
EState_VSA10
EState_VSA11
EState_VSA2
EState_VSA3
EState_VSA4
EState_VSA5
EState_VSA6
EState_VSA7
EState_VSA8
FpDensityMorgan1
FpDensityMorgan3
FractionCSP3
HallKierAlpha
Kappa2
Kappa3
MaxAbsEStateIndex
MinAbsEStateIndex
MinPartialCharge
MolLogP
NHOHCount
NOCount
NumAliphaticCarbocycles
NumAliphaticHeterocycles
NumAromaticCarbocycles
NumAromaticHeterocycles
NumHAcceptors
NumRotatableBonds
NumSaturatedCarbocycles
NumSaturatedRings
PEOE_VSA10
PEOE_VSA12
PEOE_VSA13
PEOE_VSA14
PEOE_VSA2
PEOE_VSA4
PEOE_VSA5
PEOE_VSA6
PEOE_VSA8
PEOE_VSA9
SMR_VSA2
SMR_VSA4
SMR_VSA5
SMR_VSA6
SMR_VSA9
SlogP_VSA1
SlogP_VSA10
SlogP_VSA11
SlogP_VSA12
SlogP_VSA3
SlogP_VSA4
SlogP_VSA5
SlogP_VSA7
SlogP_VSA8
VSA_EState10
VSA_EState2
VSA_EState3
VSA_EState4
VSA_EState5
VSA_EState6
VSA_EState7
VSA_EState8
fr_Al_COO
fr_Al_OH
fr_ArN
fr_Ar_COO
fr_Ar_NH
fr_Ar_OH
fr_C_S
fr_HOCCN
fr

## Ensuring that predictions were not from among the labeled dataet

In [7]:
data_w_target[
    data_w_target["Name"].str.contains("Ginkgetin", na=False)
]

,Name,senolytic,Library,Source,SMILES,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,...,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,qed


## Ensuring that total data in clusters = total data in dataset

In [8]:
i = 0
while i < 13:
    print(len(full_df[full_df["Cluster"] == i]))
    i = i + 1

925
917
19
1241
46
5
884
513
137
253
1265
373
215


Everything looks good

# Conclusion
Given the data imbalance problem and the problem that our testing data is unlabeled, I figured using a clustering algorithm would be more useful for this project. This is due to its unsupervised structure. The design of this multi model approach is to demonstrate the way in which clustering could be used to improve our predictions. The logisitic regression model could be replaced with any model as long as that model also predicted log-odds (This is better geometrically for the clustering). If a better model is found than the logistic regression model then I would be happy to make a second version of this file reimplementing the code with some of the same strategies as it would only improve the output further. The thought process is simple. Using the clustering method, we can reduce the space that we are predicting in to be more senolytic dense. Logically, this increases the likelihood that a value predicted to be a senolytic in this space would actually be a senolytic. It also reduces the search space for when we begin testing the possible senolytic values. For example we have 6660 data points, and our model only predicts 5 percent of these to be possible senolytics. Thats 333. If we have a cluster of 200 senolytic dense data points then 5 percent of these will be 10 values. This is just easier to search through, while also increasing the odds of our guesses being correct. My third best guess Ginkgetin is actually a senolytic that was not labeled. So I think this strategy might have some value to it.